# 阶段 1 / 阶段 2 验证：自定义 IP 是否真的进了视频流 + AXI-Lite 寄存器读写

这一个 notebook 覆盖两个阶段，运行时会自动判断当前 overlay 处于哪个阶段：

* **阶段 1**（IP 只有 AXIS，没有 AXI-Lite）：验证画面和插 IP 之前完全一致。
* **阶段 2**（IP 已经有 AXI-Lite）：额外验证 VERSION / MODE_SEL / FRAME_CNT。

In [ ]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp

%matplotlib inline

OUR_IP = "axis_video_proc_0"

ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(ROOT)

vs.stop_all_channels(vdma0, vdma1)

## 检查 1：IP 有没有被综合进 overlay

阶段 1 的 IP 还没有 AXI-Lite，所以**不会**出现在 `ol.ip_dict` 里，
必须直接看 `.hwh`。这是判断“我的 HDL 到底有没有进 bitstream”的唯一可靠方法。

In [ ]:
modules = vs.hwh_find_modules(ROOT)
in_hwh = OUR_IP in modules
in_ipdict = OUR_IP in ol.ip_dict

print("overlay 里模块总数:", len(modules))
print("%s 在 .hwh 里           :" % OUR_IP, in_hwh)
print("%s 在 ip_dict(AXI-Lite) 里:" % OUR_IP, in_ipdict)

if not in_hwh:
    raise RuntimeError("HDL 没进 overlay：检查 insert_axis_video_proc.tcl 是否跑成功、"
                       "以及生成的 .bit/.hwh 是否覆盖到了 overlay/ 目录")

## 检查 2：画面必须和插 IP 之前一模一样

启动采集通道抓一帧。抓得到、且画面正常，才说明 `demosaic -> IP -> VDMA0` 这条链是通的。

In [ ]:
vp.start_capture(vdma0, mode)

raw = vp.capture_frame(vdma0)
rgb = vp.frame_to_rgb(raw)
print("frame shape:", raw.shape, " max:", int(raw.max()))
if int(raw.max()) == 0:
    raise RuntimeError("抓到全黑帧：视频流断了，重点检查 IP 的 tready/tlast 连接")

plt.figure(figsize=(10, 6))
plt.imshow(rgb)
plt.title("stage1: video through axis_video_proc")
plt.axis("off")
plt.show()

## 检查 3：AXI-Lite 寄存器（阶段 2 才有效）

需要满足：VERSION 正确、MODE_SEL 可读回、FRAME_CNT 在递增、写未映射地址不死机。

In [ ]:
if not in_ipdict:
    print("阶段 1：" + OUR_IP + " 还没有 AXI-Lite 从接口。")
    print("画面检查已通过；寄存器测试要等阶段 2 加完 s_axi 后重跑本单元。")
else:
    from lib.ai_video_regs import AiVideoRegs
    regs = AiVideoRegs(ol.axis_video_proc_0)

    print("1) VERSION      = 0x%08X" % regs.version())

    regs.set_mode(1)
    got = regs.mode_sel()
    print("2) MODE_SEL 写 1 读回:", got)
    assert got == 1, "MODE_SEL 读回不对"
    regs.set_mode(0)

    f0 = regs.frame_cnt()
    time.sleep(0.5)
    f1 = regs.frame_cnt()
    print("3) FRAME_CNT   : %d -> %d" % (f0, f1))
    assert f1 != f0, "FRAME_CNT 没递增：视频流没在跑（先确认采集通道已启动）"

    try:
        regs.wr(0x100, 0xDEADBEEF)
        print("4) 写未映射地址 0x100 未抛异常")
    except Exception as exc:
        print("4) 写未映射地址抛异常（可接受）:", exc)
    print("   之后仍能读 VERSION: 0x%08X" % regs.version())

    print("阶段 2 寄存器测试通过")

## 结论

* 检查 1、2 通过 → 阶段 1 完成。
* 检查 3 也通过 → 阶段 2 完成，可以进入阶段 3（特征提取）。
* 抓不到帧 / 全黑 → 先回 `verify.md` 的“常见失败与处理”表，不要继续往下做。